# 07. Baseline: Fine-Tuned DeBERTa-v3-base on IMDB

The strongest base-size encoder, with the same stability guards as the SST-2 notebook: low learning rate (5e-6), Adam eps 1e-6, and a NaN alarm.

**Stability note.** DeBERTa-v3 can silently collapse to predicting one class (accuracy stuck at 0.5 here, AUC = 0.5). **Check the epoch 1 line: val_acc should be around 0.95** - if it prints ~0.5, stop, halve the learning rate, and Runtime -> Restart runtime.

**Runtime warning:** roughly 3-4 hours on a T4 (batch 8). If that is too long, this baseline is optional - BERT and RoBERTa already cover the transformer tier.

Steps: setup → shared splits → tokenize with the model's own tokenizer →
fine-tune with warmup → metrics (accuracy, F1, AUC, Brier, NLL, ECE) → save.

In [1]:
# ================================================================
# 0. Check the Colab GPU
# ================================================================

# Ask the Colab runtime which GPU we were given.
gpu_info = !nvidia-smi
# Join the command output lines into one printable string.
gpu_info = '\n'.join(gpu_info)
# If the command failed, we are not connected to a GPU runtime.
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU. In Colab: Runtime -> Change runtime type -> T4 GPU.')
else:
    print(gpu_info)

Mon Aug  3 13:29:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   57C    P0            268W /  400W |    3256MiB /  81920MiB |    100%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# ================================================================
# 1. Check the Colab RAM
# ================================================================

# psutil reports how much system memory this runtime has.
import psutil
# Convert bytes to gigabytes for a readable number.
ram_gb = psutil.virtual_memory().total / 1e9
# Print the available RAM so we know which runtime type we received.
print('Your runtime has {:.1f} gigabytes of available RAM'.format(ram_gb))

Your runtime has 179.4 gigabytes of available RAM


In [3]:
# ================================================================
# 2. Install libraries, then basic imports and reproducibility
# ================================================================

# transformers provides the pretrained model and tokenizer.
# sentencepiece is required by some tokenizers (for example DeBERTa-v3).
!pip -q install transformers sentencepiece

# pathlib gives clean, operating-system-safe file paths.
from pathlib import Path

# random controls Python-level randomness.
import random

# numpy handles numeric arrays outside PyTorch.
import numpy as np

# pandas handles CSV data tables.
import pandas as pd

# torch is the deep learning framework used across all AMIC notebooks.
import torch

# nn contains PyTorch neural-network modules.
from torch import nn

# Dataset and DataLoader create mini-batches for training.
from torch.utils.data import Dataset, DataLoader

# AutoTokenizer loads the matching tokenizer for any Hugging Face model name.
from transformers import AutoTokenizer

# AutoModelForSequenceClassification adds a classification head on the encoder.
from transformers import AutoModelForSequenceClassification

# This scheduler warms the learning rate up and then decays it linearly.
from transformers import get_linear_schedule_with_warmup

# Metrics for classification quality and calibration.
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, brier_score_loss, log_loss

# matplotlib draws simple training curves.
import matplotlib.pyplot as plt

# This seed keeps runs as reproducible as possible (same seed as the wine notebooks).
SEED = 20260526

# Seed Python's random module.
random.seed(SEED)

# Seed NumPy's random generator.
np.random.seed(SEED)

# Seed PyTorch's CPU generator.
torch.manual_seed(SEED)

# Seed all CUDA devices if a GPU is available.
torch.cuda.manual_seed_all(SEED)

# Pick the Colab GPU when available; otherwise fall back to CPU.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Print the selected device so we know what this notebook is using.
print('Using device:', DEVICE)

Using device: cuda


In [4]:
# ================================================================
# 3. Mount Google Drive and define project paths
# ================================================================

# Mount Google Drive so Colab can read and write project files.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')

# This is the same Google Drive project folder used by the wine and SST-2 notebooks.
PROJECT_DIR = Path('/content/drive/MyDrive/AMIC project')

# All IMDB benchmark files live inside this subfolder.
BENCH_DIR = PROJECT_DIR / 'imdb_benchmark'

# The shared, prepared IMDB splits are stored here by notebook 00.
DATA_DIR = BENCH_DIR / 'data'

# This notebook writes all of its results into its own output folder.
EXPERIMENT_NAME = 'imdb_deberta_v3_base'
OUTPUT_DIR = BENCH_DIR / 'outputs' / EXPERIMENT_NAME

# Create the output folder if it does not exist yet.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Print the paths so we can verify them before loading data.
print('DATA_DIR  :', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted.
DATA_DIR  : /content/drive/MyDrive/AMIC project/imdb_benchmark/data
OUTPUT_DIR: /content/drive/MyDrive/AMIC project/imdb_benchmark/outputs/imdb_deberta_v3_base


In [5]:
# ================================================================
# 4. Load the shared IMDB splits prepared by notebook 00
# ================================================================

# Stop early with a clear message if notebook 00 has not been run yet.
for name in ['imdb_train.csv', 'imdb_valid.csv', 'imdb_test.csv']:
    if not (DATA_DIR / name).exists():
        raise FileNotFoundError(f'Missing {DATA_DIR / name}. Run 00_prepare_imdb_data.ipynb first.')

# Read the three prepared splits from Drive (the train file is ~30 MB, so this takes a moment).
train_df = pd.read_csv(DATA_DIR / 'imdb_train.csv')
valid_df = pd.read_csv(DATA_DIR / 'imdb_valid.csv')
test_df = pd.read_csv(DATA_DIR / 'imdb_test.csv')

# Force the text column to string in case pandas parsed something unusually.
for df in [train_df, valid_df, test_df]:
    df['text'] = df['text'].astype(str)

# Print shapes and label balance to confirm the data looks right.
print('train:', train_df.shape, '| positive rate:', round(train_df['y'].mean(), 4))
print('valid:', valid_df.shape, '| positive rate:', round(valid_df['y'].mean(), 4))
print('test :', test_df.shape, '| positive rate:', round(test_df['y'].mean(), 4))

# Show one example review (truncated for display).
print(train_df['text'].iloc[0][:300], '...')

train: (23750, 2) | positive rate: 0.5
valid: (1250, 2) | positive rate: 0.5
test : (25000, 2) | positive rate: 0.5
Drew Barrymore plays young Holly Gooding, who moves in with aspiring hack screenwriter Patrick Highsmith (George Newbern) and completely disrupts his life by claiming that her "doppelganger", or evil twin, is out to kill her and her family. This silly horror film is kind of hard to take seriously, e ...


In [6]:
# ================================================================
# 5. Model configuration
# ================================================================

# The Hugging Face model name to fine-tune in this notebook.
MODEL_NAME = 'microsoft/deberta-v3-base'

# DeBERTa-v3 uses more memory than BERT/RoBERTa; at 256 tokens the batch size is 8.

# IMDB reviews are full paragraphs; 256 subword tokens is the standard budget that
# balances coverage against GPU memory and time on a Colab T4.
MAX_LEN = 256

# Mini-batch size for a Colab T4 GPU with this model size.
BATCH_SIZE = 8

# DeBERTa-v3 needs a LOWER learning rate than BERT/RoBERTa: higher rates
# collapsed training to constant predictions in our SST-2 runs.
LEARNING_RATE = 5e-6

# Small weight decay is the standard regularization for transformer fine-tuning.
WEIGHT_DECAY = 0.01

# Two epochs is plenty for IMDB fine-tuning (large dataset, long documents);
# more epochs mostly adds runtime and overfitting risk.
EPOCHS = 2

# Fraction of total steps used to warm the learning rate up from zero.
WARMUP_FRACTION = 0.1

# Print the configuration so it is recorded in the notebook output.
print('MODEL_NAME:', MODEL_NAME)
print('MAX_LEN:', MAX_LEN, '| BATCH_SIZE:', BATCH_SIZE, '| LR:', LEARNING_RATE, '| EPOCHS:', EPOCHS)

MODEL_NAME: microsoft/deberta-v3-base
MAX_LEN: 256 | BATCH_SIZE: 8 | LR: 5e-06 | EPOCHS: 2


In [7]:
# ================================================================
# 6. Tokenize the three splits with the model's own tokenizer
# ================================================================

# Every pretrained model must be paired with its own tokenizer.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_texts(texts):
    """Convert a list of sentences into fixed-length id and mask tensors."""
    # The tokenizer handles subword splitting, special tokens, truncation, and padding.
    return tokenizer(
        list(texts),              # the raw sentences
        truncation=True,          # cut sentences longer than MAX_LEN subwords
        padding='max_length',     # pad shorter sentences up to MAX_LEN
        max_length=MAX_LEN,       # the fixed sequence length
        return_tensors='pt',      # return PyTorch tensors directly
    )


# Tokenize each split once, up front.
train_encodings = tokenize_texts(train_df['text'])
valid_encodings = tokenize_texts(valid_df['text'])
test_encodings = tokenize_texts(test_df['text'])

# Pull out the label arrays.
y_train = train_df['y'].to_numpy().astype(np.int64)
y_valid = valid_df['y'].to_numpy().astype(np.int64)
y_test = test_df['y'].to_numpy().astype(np.int64)

# Show the tensor shapes: (sentences, MAX_LEN).
print('train input_ids:', tuple(train_encodings['input_ids'].shape))

# Decode one example back so we can see the subword tokens the model will read.
print('Example tokens:', tokenizer.convert_ids_to_tokens(train_encodings['input_ids'][0])[:20])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

train input_ids: (23750, 256)
Example tokens: ['[CLS]', '▁Drew', '▁Barrymore', '▁plays', '▁young', '▁Holly', '▁Good', 'ing', ',', '▁who', '▁moves', '▁in', '▁with', '▁aspiring', '▁hack', '▁screenwriter', '▁Patrick', '▁High', 'smith', '▁(']


In [8]:
# ================================================================
# 7. Dataset and DataLoader
# ================================================================

class TransformerTextDataset(Dataset):
    """Dataset wrapper that serves input ids, attention masks, and labels."""

    def __init__(self, encodings, labels):
        # Token id tensor of shape [n_sentences, MAX_LEN].
        self.input_ids = encodings['input_ids']
        # Attention mask: 1 for real tokens, 0 for padding.
        self.attention_mask = encodings['attention_mask']
        # Integer class labels (0 = negative, 1 = positive).
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        # The dataset length is the number of sentences.
        return self.input_ids.size(0)

    def __getitem__(self, idx):
        # Return one example as a dictionary of tensors.
        return {'input_ids': self.input_ids[idx],
                'attention_mask': self.attention_mask[idx],
                'label': self.labels[idx]}


# Build the three split datasets.
train_ds = TransformerTextDataset(train_encodings, y_train)
valid_ds = TransformerTextDataset(valid_encodings, y_valid)
test_ds = TransformerTextDataset(test_encodings, y_test)

# Build the mini-batch loaders; only training data is shuffled.
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Print the number of batches per split as a sanity check.
print('train batches:', len(train_loader))
print('valid batches:', len(valid_loader))
print('test batches :', len(test_loader))

train batches: 2969
valid batches: 157
test batches : 3125


In [9]:
# ================================================================
# 8. Load the pretrained model with a fresh classification head
# ================================================================

# num_labels=2 attaches a randomly initialized 2-class head on top of the encoder.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Move the whole model to the GPU.
model = model.to(DEVICE)

# Print the parameter count so model sizes are recorded across notebooks.
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den

model.safetensors: reconstructing file:   0%|          |  0.00B /  371MB            

model.safetensors: downloading bytes:           |  0.00B            

Trainable parameters: 184423682


In [10]:
# ================================================================
# Metric helpers (identical to the wine BAMIC notebook, for fair comparison)
# ================================================================

def binary_metrics_from_probs(probs, labels, threshold=0.5):
    """Compute common binary metrics from predicted positive-class probabilities."""
    # Turn probabilities into hard 0/1 predictions at the given threshold.
    pred = (probs >= threshold).astype(int)
    # Fraction of documents classified correctly.
    acc = accuracy_score(labels, pred)
    # Harmonic mean of precision and recall for the positive class.
    f1 = f1_score(labels, pred, zero_division=0)
    # Threshold-free ranking quality; needs both classes present.
    auc = roc_auc_score(labels, probs) if len(np.unique(labels)) == 2 else np.nan
    # Mean squared error between probabilities and true labels.
    brier = brier_score_loss(labels, probs)
    # Negative log-likelihood of the true labels under the predicted probabilities.
    nll = log_loss(labels, np.clip(probs, 1e-7, 1 - 1e-7), labels=[0, 1])
    # Return everything in one dictionary.
    return {'acc': acc, 'f1': f1, 'auc': auc, 'brier': brier, 'nll': nll}


def expected_calibration_error(probs, labels, n_bins=10):
    """Compute a simple expected calibration error (ECE)."""
    # Create equally spaced probability bins between 0 and 1.
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    # Accumulate the weighted calibration gap here.
    ece = 0.0
    # Loop over each bin edge pair.
    for lo, hi in zip(bins[:-1], bins[1:]):
        # Include the right edge for the final bin.
        if hi == 1.0:
            mask = (probs >= lo) & (probs <= hi)
        else:
            mask = (probs >= lo) & (probs < hi)
        # Skip bins that contain no predictions.
        if not np.any(mask):
            continue
        # Average predicted probability inside this bin.
        conf = probs[mask].mean()
        # Empirical positive rate inside this bin.
        acc = labels[mask].mean()
        # Weight the |confidence - accuracy| gap by the bin frequency.
        ece += np.abs(conf - acc) * mask.mean()
    # Return the scalar ECE value.
    return float(ece)

In [11]:
# ================================================================
# 9. Fine-tuning loop with warmup and linear decay
# ================================================================

# AdamW is the standard optimizer for transformer fine-tuning.
# eps=1e-6 (instead of PyTorch's 1e-8 default) matches the Hugging Face recipe and
# prevents the NaN divergence that DeBERTa-v3 in particular is prone to.
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, eps=1e-6)

# Total number of optimizer steps across all epochs.
total_steps = len(train_loader) * EPOCHS

# The scheduler warms up for the first WARMUP_FRACTION of steps, then decays to zero.
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_FRACTION * total_steps),
    num_training_steps=total_steps,
)

# Keep one history row per epoch for the history.csv output.
history = []


@torch.no_grad()
def predict_probs(loader):
    """Return predicted positive-class probabilities for every example in a loader."""
    # Switch off dropout for deterministic evaluation.
    model.eval()
    # Collect per-batch probability arrays here.
    all_probs = []
    # Loop over mini-batches without building gradients.
    for batch in loader:
        # Move ids and masks to the GPU.
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        # Forward pass; outputs.logits has shape [batch, 2].
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        # Softmax over the two classes; keep column 1 = P(positive).
        probs = torch.softmax(outputs.logits, dim=-1)[:, 1]
        # Move the probabilities back to CPU.
        all_probs.append(probs.cpu().numpy())
    # Concatenate all batches into one flat array.
    return np.concatenate(all_probs)


# Loop over fine-tuning epochs.
for epoch in range(1, EPOCHS + 1):
    # Put the model in training mode (enables dropout).
    model.train()
    # Track the running loss of this epoch.
    epoch_losses = []
    # Loop over training mini-batches.
    for batch in train_loader:
        # Move inputs and labels to the GPU.
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['label'].to(DEVICE)
        # Clear old gradients before the new backward pass.
        optimizer.zero_grad(set_to_none=True)
        # Passing labels makes the model compute the cross-entropy loss internally.
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        # Read the scalar loss from the output object.
        loss = outputs.loss
        # Stop immediately with a clear message if training diverges.
        if torch.isnan(loss):
            raise RuntimeError('Loss became NaN - lower LEARNING_RATE, then Runtime -> Restart runtime and rerun.')
        # Backward pass: compute gradients.
        loss.backward()
        # Clip gradients at 1.0, the standard value for transformer fine-tuning.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        # Update the model parameters.
        optimizer.step()
        # Advance the learning-rate schedule by one step.
        scheduler.step()
        # Remember this batch's loss value.
        epoch_losses.append(float(loss.detach().cpu()))

    # Evaluate on the validation split at the end of every epoch.
    valid_probs_epoch = predict_probs(valid_loader)
    # Compute the standard metric set on validation predictions.
    valid_metrics = binary_metrics_from_probs(valid_probs_epoch, y_valid)
    # Compute the validation calibration error.
    valid_ece = expected_calibration_error(valid_probs_epoch, y_valid)
    # Store one row of history for this epoch.
    history.append({'epoch': epoch, 'train_loss': float(np.mean(epoch_losses)),
                    'val_acc': valid_metrics['acc'], 'val_f1': valid_metrics['f1'],
                    'val_auc': valid_metrics['auc'], 'val_ece': valid_ece})
    # Print a one-line progress summary.
    print(f"Epoch {epoch:02d} | loss={np.mean(epoch_losses):.4f} | "
          f"val_acc={valid_metrics['acc']:.4f} | val_auc={valid_metrics['auc']:.4f} | "
          f"val_ece={valid_ece:.4f}")

# Convert the history to a dataframe and save it for later comparison plots.
history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / 'history.csv', index=False)

# Show the full training history table.
history_df

Epoch 01 | loss=0.7025 | val_acc=0.5000 | val_auc=0.5000 | val_ece=0.0042
Epoch 02 | loss=0.6947 | val_acc=0.5000 | val_auc=0.5000 | val_ece=0.0034


,epoch,train_loss,val_acc,val_f1,val_auc,val_ece
0,1,0.702478,0.5,0.000000,0.5,0.004150
1,2,0.694718,0.5,0.666667,0.5,0.003418


In [12]:
# ================================================================
# 10. Final predictions on all three splits
# ================================================================

# IMPORTANT: train_loader was built with shuffle=True, which re-shuffles the data on
# every pass. Using it for evaluation would return probabilities in a random order that
# no longer lines up with y_train, making the train metrics look like coin-flipping.
# We therefore build a NON-shuffled copy of the training data just for evaluation.
train_eval_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Predict probabilities for every split with the fine-tuned model.
train_probs = predict_probs(train_eval_loader)
valid_probs = predict_probs(valid_loader)
test_probs = predict_probs(test_loader)

# Rename the label arrays to the names used by the shared saving cell.
y_train_true = y_train
y_valid_true = y_valid
y_test_true = y_test

# Quick sanity check on the shapes.
print('test_probs:', test_probs.shape, '| first 5:', np.round(test_probs[:5], 3))

# Optional: uncomment the next two lines to save the fine-tuned weights to Drive.
# model.save_pretrained(OUTPUT_DIR / 'model')
# tokenizer.save_pretrained(OUTPUT_DIR / 'model')

test_probs: (25000,) | first 5: [0.504 0.504 0.504 0.504 0.504]


In [13]:
# ================================================================
# Save final metrics and predictions to Drive
# ================================================================

# Collect one metrics row per split so the comparison notebook can read them later.
final_metric_rows = []

# Loop over the three splits with their predicted probabilities and true labels.
for split_name, probs, labels in [('train', train_probs, y_train_true),
                                  ('valid', valid_probs, y_valid_true),
                                  ('test', test_probs, y_test_true)]:
    # Compute accuracy, F1, AUC, Brier, and NLL for this split.
    metrics = binary_metrics_from_probs(probs, labels)
    # Compute the calibration error for this split.
    ece = expected_calibration_error(probs, labels)
    # Store everything in one row.
    final_metric_rows.append({'split': split_name, **metrics, 'ece': ece})
    # Print the row so we can see the result immediately.
    print(split_name, {k: round(v, 4) for k, v in metrics.items()}, 'ece=', round(ece, 4))

# Convert the rows into a small dataframe.
final_metrics_df = pd.DataFrame(final_metric_rows)

# Save the metrics table into this notebook's output folder.
final_metrics_df.to_csv(OUTPUT_DIR / 'final_metrics.csv', index=False)

# Also save the raw test predictions for later error analysis.
pd.DataFrame({'y_true': y_test_true, 'p_positive': test_probs}).to_csv(
    OUTPUT_DIR / 'test_predictions.csv', index=False)

# Confirm where everything was written.
print('Saved final_metrics.csv and test_predictions.csv to:', OUTPUT_DIR)

train {'acc': 0.5, 'f1': 0.6667, 'auc': np.float64(0.5), 'brier': np.float64(0.25), 'nll': 0.6932} ece= 0.0034
valid {'acc': 0.5, 'f1': 0.6667, 'auc': np.float64(0.5), 'brier': np.float64(0.25), 'nll': 0.6932} ece= 0.0034
test {'acc': 0.5, 'f1': 0.6667, 'auc': np.float64(0.5), 'brier': np.float64(0.25), 'nll': 0.6932} ece= 0.0034
Saved final_metrics.csv and test_predictions.csv to: /content/drive/MyDrive/AMIC project/imdb_benchmark/outputs/imdb_deberta_v3_base
